# 面试问题：PagedAttention 怎样管理 KV Cache？块表、动态增长、共享和 Copy-on-Write 如何实现？

**一句话回答**：把每条序列的逻辑 KV token 映射到固定大小物理 block，按需分配而不是连续预留最大长度；attention kernel 通过 block table gather 对应 K/V。前缀或 beam 可共享只读 block，首次写入共享尾块前执行 Copy-on-Write，并用引用计数安全回收。

本 Notebook 手写 block pool、序列表、append/gather、分页 attention、fork/COW、回收、准入和碎片指标，不调用推理服务框架。


In [ ]:
from collections import OrderedDict
import math
import numpy as np

SEED131=13101; rng131=np.random.default_rng(SEED131)
assert SEED131==13101
assert math.ceil(9/4)==3
assert np.isfinite(rng131.normal())


## 1. 连续预留会产生内部和外部碎片

请求输出长度未知，若按 `max_tokens` 连续预留会大量空置；若按当前长度频繁扩容，又需要搬迁并产生不可合并空洞。分页把浪费限制在每条活跃序列最后一个 block，并允许物理 block 非连续。


In [ ]:
lengths131=[3,9,10]; maxima131=[16,16,32]; block131=4
reserved_waste131=sum(m-l for l,m in zip(lengths131,maxima131))
paged_waste131=sum(math.ceil(l/block131)*block131-l for l in lengths131)
assert reserved_waste131==42
assert paged_waste131==6
assert paged_waste131<reserved_waste131


## 2. BlockPool 负责唯一 owner 与引用计数

空闲表分配物理 block，`retain/release` 管理共享。double free、耗尽和未知 block 必须显式失败；否则一个序列可能覆盖另一个序列仍在读取的 KV。生产实现还需并发原子性和设备 stream 生命周期。


In [ ]:
class BlockPool131:
    def __init__(self,n): self.free=list(range(n)); self.ref={}
    def alloc(self):
        if not self.free: raise RuntimeError("oom")
        b=self.free.pop(0); self.ref[b]=1; return b
    def retain(self,b): self.ref[b]+=1
    def release(self,b):
        self.ref[b]-=1
        if self.ref[b]==0: del self.ref[b]; self.free.append(b); self.free.sort()
pool131=BlockPool131(3); b0131=pool131.alloc(); pool131.retain(b0131); pool131.release(b0131)
assert pool131.ref[b0131]==1
assert len(pool131.free)==2
assert b0131 not in pool131.free


## 3. Sequence Page Table 映射逻辑 token 到物理位置

每新增 token，若进入新逻辑 block 就分配物理 block；物理地址由 `table[token//B]` 与 `token%B` 组成。元数据更新与 KV 写入需要有提交顺序，避免 crash 后表指向未完整写入的数据。


In [ ]:
class Seq131:
    def __init__(self,pool,B=4): self.pool=pool; self.B=B; self.table=[]; self.length=0; self.data={}
    def append(self,k,v):
        if self.length%self.B==0: self.table.append(self.pool.alloc())
        b=self.table[self.length//self.B]; off=self.length%self.B; self.data[(b,off)]=(np.array(k),np.array(v)); self.length+=1
    def address(self,i):
        if not 0<=i<self.length: raise IndexError(i)
        return self.table[i//self.B],i%self.B
seq131=Seq131(BlockPool131(8)); [seq131.append([i,1],[i,-1]) for i in range(6)]
assert len(seq131.table)==2
assert seq131.address(5)==(seq131.table[1],1)
assert seq131.length==6


## 4. PagedAttention 改变寻址，不改变 attention 数学

kernel 按 block table gather 历史 K/V，再计算新 query 对所有合法 token 的注意力。教学实现先重组连续数组作为 oracle；真实 kernel 会直接遍历物理 blocks，避免额外 copy。必须只读到 `length`，不能把尾块未初始化槽位纳入 softmax。


In [ ]:
def gather_kv131(seq):
    pairs=[seq.data[seq.address(i)] for i in range(seq.length)]; return np.stack([x[0] for x in pairs]),np.stack([x[1] for x in pairs])
def decode_attn131(q,seq):
    K,V=gather_kv131(seq); s=K@q/math.sqrt(len(q)); p=np.exp(s-s.max()); p/=p.sum(); return p@V
K131,V131=gather_kv131(seq131); out131=decode_attn131(np.array([1.,.5]),seq131)
assert K131.shape==(6,2) and V131.shape==(6,2)
assert out131.shape==(2,)
assert np.all(np.isfinite(out131))


## 5. Beam/前缀共享需要 Copy-on-Write

fork 时共享已完成 blocks 并增加引用计数。若最后一个 block 未满，子序列追加会修改共享物理页，必须先复制有效槽位到新 block；完整历史 block 可继续只读共享。COW 的判断发生在写前，而不是写坏之后再修复。


In [ ]:
def fork131(parent):
    child=Seq131(parent.pool,parent.B); child.table=list(parent.table); child.length=parent.length; child.data=parent.data
    for b in child.table: child.pool.retain(b)
    return child
def ensure_private_tail131(seq):
    if seq.length%seq.B and seq.pool.ref[seq.table[-1]]>1:
        old=seq.table[-1]; new=seq.pool.alloc()
        for off in range(seq.length%seq.B): seq.data[(new,off)]=tuple(x.copy() for x in seq.data[(old,off)])
        seq.table[-1]=new; seq.pool.release(old)
pool_cow131=BlockPool131(8); parent131=Seq131(pool_cow131); [parent131.append([i],[i]) for i in range(3)]; child131=fork131(parent131); ensure_private_tail131(child131)
assert parent131.table[-1]!=child131.table[-1]
assert pool_cow131.ref[parent131.table[-1]]==1
assert child131.data[child131.address(2)][0][0]==2


## 6. 完成、取消与超时都要走同一释放路径

序列结束后逐 block `release`，只有 refcount 归零才回空闲表。取消请求还要等待正在使用 KV 的 GPU work 安全结束。幂等 cleanup 防止重试 double free；泄漏指标应按 request 和 block age 追踪。


In [ ]:
def cleanup131(seq):
    for b in list(seq.table): seq.pool.release(b)
    seq.table=[]; seq.length=0
shared_block131=parent131.table[0]; cleanup131(child131)
assert shared_block131 in pool_cow131.ref
cleanup131(parent131)
assert shared_block131 in pool_cow131.free
assert not parent131.table and parent131.length==0


## 7. 准入按可增长预算，不只看当前占用

decode 每步都可能申请新 block。调度器应保留安全水位，根据请求最大剩余 token 估算最坏需求；内存紧张时选择延迟准入、抢占可恢复序列或降级长度，不能在 kernel 中途才随机 OOM。


In [ ]:
def blocks_needed131(current,max_new,B): return math.ceil((current+max_new)/B)-math.ceil(current/B)
def admit131(free,current,max_new,B,reserve): return blocks_needed131(current,max_new,B)<=free-reserve
assert blocks_needed131(7,6,4)==2
assert admit131(5,7,6,4,2)
assert not admit131(3,7,6,4,2)


## 8. 同时评测正确性、利用率和调度收益

正确性比较连续与分页 attention；系统指标包括有效 token slots/已分配 slots、COW 次数、block OOM、抢占率、batch size、TTFT、TPOT 和吞吐。block 太小元数据与寻址开销高，太大尾块浪费高，需要在真实长度分布上扫描。


In [ ]:
def utilization131(lengths,B): return sum(lengths)/(sum(math.ceil(x/B)*B for x in lengths) or 1)
u4_131=utilization131([3,9,10],4); u16_131=utilization131([3,9,10],16)
assert 0<u4_131<=1
assert u4_131>u16_131
assert math.isclose(u4_131,22/28)


## 面试总结

完整链路是：**固定 block pool → per-sequence block table → append 时按需分配 → kernel 按逻辑位置寻址 → 尾槽绝不参与 softmax → fork 墯加 refcount → 共享尾块写前 COW → 完成/取消幂等释放 → 按增长预算准入 → 扫 block size 的利用率与延迟**。PagedAttention 解决动态 KV 内存管理，不能单靠它保证调度公平或模型正确性。

延伸阅读：[vLLM / PagedAttention](https://arxiv.org/abs/2309.06180)、[vAttention](https://arxiv.org/abs/2405.04437)、[Orca](https://www.usenix.org/conference/osdi22/presentation/yu)。
